### Earthquake PINN Training on Colab GPU
This notebook clones the repository, installs dependencies, and runs the full-scale training.
It also includes optional Hyperparameter Optimization (Optuna).

**Note**: Make sure you have pushed your local changes (`git push origin dev`) before running this.

In [ ]:
!nvidia-smi

In [ ]:
# Install uv and clean up existing clone
!pip install uv
import os
%cd /content
!rm -rf earthquake_proj
!git clone -b dev --single-branch https://github.com/sattary/earthquake_proj.git
%cd earthquake_proj

In [ ]:
# Sync environment
!uv sync

In [ ]:
!ls

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figs", exist_ok=True)

# Ensure package initialization for -m calls
if os.path.exists('src'):
    if not os.path.exists('src/__init__.py'):
        open('src/__init__.py', 'a').close()
        print("Initialized 'src' as package.")
elif os.path.exists('earthquake_proj'):
    if not os.path.exists('earthquake_proj/__init__.py'):
        open('earthquake_proj/__init__.py', 'a').close()
        print("Initialized 'earthquake_proj' as package.")

In [ ]:
# Step 1: Tune Hyperparameters (Optional - skip if using defaults)
!PYTHONPATH=. uv run python -m src.pipelines.cli tune --trials 50 --epochs 500


In [ ]:
# Step 2: Final Stabilized Training
!PYTHONPATH=. uv run python -m src.pipelines.cli train --epochs 20000 --n-coll 20000


In [ ]:
# Step 3: Generate Academic Figure Suite (PDF, SVG, High-Res PNG)
!PYTHONPATH=. uv run python -m src.pipelines.cli results-suite


In [ ]:
from IPython.display import Image, display
import os
for f in ['results/figs/stress_map_10km.png', 'results/figs/velocity_map_10km.png']:
    if os.path.exists(f): display(Image(f))

In [ ]:
# Pack Everything for Download/Active Storage
!zip -r results_archive.zip results/ checkpoints/final_model.pth
try:
  from google.colab import files
  files.download('results_archive.zip')
except ImportError:
  print('Not running on Google Colab, skipping download.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls


In [ ]:
cp -R results_archive.zip /content/drive/MyDrive/

In [ ]:
!ls /content/drive/MyDrive/ | grep .zip